In [8]:
import pandas as pd
import numpy as np

In [9]:
# Load raw dataset
df = pd.read_csv("../data/raw/intern_performance_dataset.csv")

# Preview data
df.head()

,Intern_ID,Name,Gender,Domain,College,Internship_Start_Date,Week_Number,Week_Start_Date,Tasks_Assigned,Tasks_Completed,Task_Completion_Rate (%),Attendance (%),Late_Submissions,Mentor_Rating (1-5),Performance_Score
0,1,Rishi Mishra,Male,AI,IIT,8/16/2024,1,8/16/2024,15,5,33.33,76.91,0,4.06,60.77
1,1,Rishi Mishra,Male,AI,IIT,8/16/2024,2,8/16/2024,15,13,86.67,83.35,3,4.55,86.97
2,1,Rishi Mishra,Male,AI,IIT,8/16/2024,3,8/16/2024,14,4,28.57,94.35,5,2.76,56.29
3,1,Rishi Mishra,Male,AI,IIT,8/16/2024,4,8/16/2024,5,5,100.00,99.70,3,3.20,89.11
4,1,Rishi Mishra,Male,AI,IIT,8/16/2024,5,8/16/2024,13,5,38.46,79.07,0,2.53,54.28


In [10]:
df.shape, df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Intern_ID                 6000 non-null   int64  
 1   Name                      6000 non-null   object 
 2   Gender                    6000 non-null   object 
 3   Domain                    6000 non-null   object 
 4   College                   6000 non-null   object 
 5   Internship_Start_Date     6000 non-null   object 
 6   Week_Number               6000 non-null   int64  
 7   Week_Start_Date           6000 non-null   object 
 8   Tasks_Assigned            6000 non-null   int64  
 9   Tasks_Completed           6000 non-null   int64  
 10  Task_Completion_Rate (%)  6000 non-null   float64
 11  Attendance (%)            6000 non-null   float64
 12  Late_Submissions          6000 non-null   int64  
 13  Mentor_Rating (1-5)       6000 non-null   float64
 14  Performa

((6000, 15), None)

In [11]:
df.columns = [
    "intern_id",
    "name",
    "gender",
    "domain",
    "college",
    "internship_start_date",
    "week_number",
    "week_start_date",
    "tasks_assigned",
    "tasks_completed",
    "task_completion_rate",
    "attendance",
    "late_submissions",
    "mentor_rating",
    "performance_score"
]

In [12]:
df["internship_start_date"] = pd.to_datetime(df["internship_start_date"])
df["week_start_date"] = pd.to_datetime(df["week_start_date"])

In [14]:
# Check missing values
df.isnull().sum()
# Fill missing numeric values with median
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [15]:
df = df.drop_duplicates()

In [16]:
# Tasks completed should not exceed tasks assigned
df = df[df["tasks_completed"] <= df["tasks_assigned"]]

# Clip percentage-based columns
df["attendance"] = df["attendance"].clip(0, 100)
df["task_completion_rate"] = df["task_completion_rate"].clip(0, 100)

# Mentor rating must be between 1 and 5
df["mentor_rating"] = df["mentor_rating"].clip(1, 5)

In [19]:
intern_level_df = df.groupby(
    ["intern_id", "name", "gender", "domain", "college"]
).agg(
    avg_attendance=("attendance", "mean"),
    avg_task_completion=("task_completion_rate", "mean"),
    total_tasks_assigned=("tasks_assigned", "sum"),
    total_tasks_completed=("tasks_completed", "sum"),
    avg_mentor_rating=("mentor_rating", "mean"),
    total_late_submissions=("late_submissions", "sum"),
    weeks_completed=("week_number", "nunique")
).reset_index()

In [20]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

cols = ["avg_attendance", "avg_task_completion", "avg_mentor_rating"]

intern_level_df[cols] = scaler.fit_transform(intern_level_df[cols])

intern_level_df["performance_index"] = (
    0.30 * intern_level_df["avg_task_completion"] +
    0.30 * intern_level_df["avg_attendance"] +
    0.25 * intern_level_df["avg_mentor_rating"] -
    0.15 * (intern_level_df["total_late_submissions"] / intern_level_df["weeks_completed"])
)

In [21]:
df.to_csv("../data/processed/intern_weekly_cleaned.csv", index=False)
intern_level_df.to_csv("../data/processed/intern_level_cleaned.csv", index=False)

In [22]:
import os
os.listdir("../data/processed")

['.ipynb_checkpoints', 'intern_level_cleaned.csv', 'intern_weekly_cleaned.csv']